Capstone Agent: Cease & Desist Document Processing Agent

In [1]:
#Install required packages for the Colab session.
!pip install langgraph langchain langchain-community pymupdf opencv-python pytesseract
!pip install langchain-openai langchain-core huggingface_hub langchain-groq
!pip install pypdf transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

In [21]:
import os, io, json, csv, sqlite3, datetime, fitz, pytesseract, cv2, numpy as np
from google.colab import files
from PIL import Image
from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal
#from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from google.colab import userdata
from langchain_groq import ChatGroq
#from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
#os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
#from datetime import datetime
from datetime import datetime,date
#from langchain_community.llms import HuggingFaceHub
#os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get("HUGGINGFACEHUB_API_TOKEN")
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
# if "GROQ_API_KEY" not in os.environ:
#     os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
print("Audit file path:", os.path.abspath("audit.txt"))
print("Database file path:", os.path.abspath("cease_request.db"))

Audit file path: /content/audit.txt
Database file path: /content/cease_request.db


In [22]:
# -------------------------
# Define State
# -------------------------
class AgentState(TypedDict):
    document_name: str
    document_path: str
    document_text: str
    classification: Literal["Cease", "Uncertain", "Irrelevant"]
    confidence: float
    explanation: str
    audit_log: list

In [23]:
# -------------------------
# Initialize LLM
# -------------------------

llm = ChatGroq(
    model="openai/gpt-oss-120b",
#     model="qwen/qwen3-32b",
    temperature=0,
    max_retries=2,
)
#llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
#llm = ChatOpenAI(model="gpt-4", temperature=0)
prompt = ChatPromptTemplate.from_template("""
You are a compliance assistant. Classify the following document into one of three categories:
- Cease: Valid cease & desist request
- Uncertain: Requires manual review
- Irrelevant: Not a cease request

Document:
{text}

Return the category name only, followed by a short explanation.
""")

In [24]:
# -------------------------
# Preprocessing for OCR
# -------------------------
def preprocess_image_for_ocr(pixmap_bytes: bytes):
    """Preprocess image: convert, deskew, binarize."""
    nparr = np.frombuffer(pixmap_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_GRAYSCALE)

    # Thresholding (binarization)
    _, thresh = cv2.threshold(img, 150, 255, cv2.THRESH_BINARY)

    # Deskew using image moments
    coords = np.column_stack(np.where(thresh > 0))
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle

    (h, w) = img.shape[:2]
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    deskewed = cv2.warpAffine(img, M, (w, h),
                              flags=cv2.INTER_CUBIC,
                              borderMode=cv2.BORDER_REPLICATE)
    return deskewed

In [25]:
# -------------------------
# ingestion_agent Functions
# -------------------------
def find_document_path(doc_name: str, search_dirs: list) -> str:
    """Search for the document in multiple directories (recursively)."""
    for directory in search_dirs:
        for root, _, files in os.walk(directory):
            if doc_name in files:
                return os.path.join(root, doc_name)
    return None
def ingestion_agent(state: dict) -> dict:
    """Extract text from PDF, fallback to OCR with preprocessing if scanned."""
    doc_text = ""
    try:
        # If direct path fails, search recursively
        if not os.path.exists(state["document_path"]):
            search_dirs = [
                "C:/Users/hp/Downloads/capstone-project/data/pdfs",
                "C:/Users/hp/Downloads",
                "C:/Users/hp/Documents",
                os.getcwd()  # current working directory
            ]
            found_path = find_document_path(state["document_name"], search_dirs)
            if found_path:
                state["document_path"] = found_path
            else:
                raise FileNotFoundError(f"File not found in any search directory: {state['document_name']}")

        pdf = fitz.open(state["document_path"])
        for page in pdf:
            text = page.get_text()
            if text.strip():
                doc_text += text
            else:
                pix = page.get_pixmap()
                deskewed_img = preprocess_image_for_ocr(pix.tobytes("png"))
                text = pytesseract.image_to_string(deskewed_img, config="--psm 6")
                doc_text += text
        pdf.close()

    except Exception as e:
        state["audit_log"].append(f"[Ingestion] Error reading PDF: {e}")

    state["document_text"] = doc_text
    if doc_text:
        state["audit_log"].append(f"[Ingestion] Extracted text from {state['document_name']}")
    return state


In [26]:
# -------------------------------
# classification_agent Functions
# -------------------------------
def classification_agent(state: AgentState) -> AgentState:
    """Use ChatOpenAI to classify document."""
    text = state["document_text"]
    chain = prompt | llm
    response = chain.invoke({"text": text})
    output = response.content.strip()

    if output.lower().startswith("cease"):
        state["classification"] = "Cease"
    elif output.lower().startswith("uncertain"):
        state["classification"] = "Uncertain"
    else:
        state["classification"] = "Irrelevant"

    state["confidence"] = 0.9  # heuristic since LLMs don’t return probabilities
    state["explanation"] = output
    state["audit_log"].append(
        f"[Classification] {state['classification']} - {state['explanation']}"
    )
    return state

In [27]:
# -------------------------------
# database_agent Functions
# -------------------------------
def database_agent(state: dict) -> dict:
    """Store Cease requests in SQLite database."""
    try:
        conn = sqlite3.connect("cease_request.db")
        cursor = conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS requests(
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                date_received TEXT,
                document_name TEXT,
                detail TEXT
            )
        """)
        cursor.execute("""
            INSERT INTO requests(date_received, document_name, detail)
            VALUES(?,?,?)
        """, (
            datetime.now().strftime("%Y-%m-%d"),
            os.path.basename(state["document_path"]),
            state["document_text"]
        ))
        conn.commit()
        conn.close()
        state["audit_log"].append(f"[Database] Stored {state['document_name']} in database")
    except Exception as e:
        state["audit_log"].append(f"[Database] Error storing document: {e}")
    return state

In [28]:
# -------------------------
# archiving_agent Functions
# -------------------------
def archiving_agent(state: AgentState) -> AgentState:
    state["audit_log"].append(
        f"[Archive] Archived Irrelevant doc: {state['document_name']} on {date.today().strftime('%Y-%m-%d')}"
    )
    return state

In [29]:
# --------------------
# hitl_agent Functions
# --------------------
def hitl_agent(state: AgentState) -> AgentState:
    """Interactive HITL step: human reviewer classifies."""
    print("\n--- HUMAN REVIEW REQUIRED ---")
    print(f"Document: {state['document_name']}")
    print("Extracted Text:\n", state["document_text"][:500], "...")  # show snippet
    print("\nClassification uncertain. Please choose:")
    print("1. Cease")
    print("2. Irrelevant")

    choice = input("Enter your choice (1/2): ").strip()
    if choice == "1":
        reviewer_decision = "Cease"
    else:
        reviewer_decision = "Irrelevant"

    state["classification"] = reviewer_decision
    state["audit_log"].append(
        f"[HITL] Human classified {state['document_name']} → {reviewer_decision}"
    )
    return state

In [30]:
# --------------------
# audit_agent Functions
# --------------------
def audit_agent(state: AgentState) -> AgentState:
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")  # fixed
    entry = {
        "timestamp": timestamp,
        "step": "Audit",
        "action": "Workflow complete",
        "details": f"Final classification: {state['classification']}"
    }

    # Write to audit file
    with open("audit.txt", "a", encoding="utf-8") as log:
        log.write(f"{entry}\n")

    state["audit_log"].append("[Audit] Entry written to audit.txt")
    return state

In [31]:

# -------------------------
# Build LangGraph Workflow
# -------------------------
workflow = StateGraph(AgentState)

workflow.add_node("ingestion", ingestion_agent)
workflow.add_node("classification", classification_agent)
workflow.add_node("database", database_agent)
workflow.add_node("archive", archiving_agent)
workflow.add_node("hitl", hitl_agent)
workflow.add_node("audit", audit_agent)

workflow.set_entry_point("ingestion")
workflow.add_edge("ingestion", "classification")

# Initial classification routing
workflow.add_conditional_edges(
    "classification",
    lambda state: state["classification"],
    {
        "Cease": "database",
        "Irrelevant": "archive",
        "Uncertain": "hitl",
    },
)

# After HITL, route based on human decision
workflow.add_conditional_edges(
    "hitl",
    lambda state: state["classification"],
    {
        "Cease": "database",
        "Irrelevant": "archive",
#        "Uncertain": "audit"  # if human still marks uncertain
    },
)

# Database/Archive → Audit → END
workflow.add_edge("database", "audit")
workflow.add_edge("archive", "audit")
workflow.add_edge("audit", END)

# -------------------------
# Run Loop for All PDFs + Summary + Export
# -------------------------
app = workflow.compile()

In [32]:
# -----------------------------
from google.colab import files

uploaded = files.upload()

summary_counts = {"Cease": 0, "Uncertain": 0, "Irrelevant": 0}
summary_details = []

for filename in uploaded.keys():
    initial_state: AgentState = {
        "document_name": filename,
        "document_path": os.path.join("/content", filename),
        "document_text": "",
        "classification": None,
        "confidence": 0.0,
        "explanation": "",
        "audit_log": []
    }

    final_state = app.invoke(initial_state)

    print("\n=== Results for:", final_state["document_name"], "===")
    print("Final Classification:", final_state["classification"])
    print("\nAudit Trail:")
    for log in final_state["audit_log"]:
        print(log)

    summary_counts[final_state["classification"]] += 1
    summary_details.append({
        "document_name": final_state["document_name"],
        "classification": final_state["classification"],
        "confidence": final_state["confidence"],
        "explanation": final_state["explanation"]
    })

# -----------------------------
# Step 10: Display Summary
# -----------------------------
print("\n=== Summary Report ===")
print("Counts:", summary_counts)
print("\nDetails:")
for detail in summary_details:
  print(f"{detail['document_name']} → {detail['classification']}")
    #print(detail)


Saving 01_copyright_infringement_photography.pdf to 01_copyright_infringement_photography (3).pdf
Saving 02_trademark_infringement_tech.pdf to 02_trademark_infringement_tech (3).pdf
Saving 03_trade_secret_misappropriation.pdf to 03_trade_secret_misappropriation (3).pdf
Saving 04_defamation_online_review.pdf to 04_defamation_online_review (2).pdf
Saving 05_patent_infringement_medical_device.pdf to 05_patent_infringement_medical_device (3).pdf
Saving 06_harassment_workplace.pdf to 06_harassment_workplace (3).pdf
Saving 07_software_license_violation.pdf to 07_software_license_violation (2).pdf
Saving 08_non_compete_violation.pdf to 08_non_compete_violation (2).pdf
Saving 09_copyright_infringement_music.pdf to 09_copyright_infringement_music (2).pdf
Saving 10_breach_of_contract_nda.pdf to 10_breach_of_contract_nda (2).pdf
Saving bw_doc_1.pdf to bw_doc_1 (2).pdf
Saving bw_doc_2.pdf to bw_doc_2 (2).pdf
Saving bw_doc_3.pdf to bw_doc_3 (2).pdf
Saving bw_doc_4.pdf to bw_doc_4 (2).pdf
Saving bw_